# 04. 이상 탐지 모델링

3가지 비지도 학습 모델을 적용하여 선박 이상 행동을 탐지한다.

| 모델 | 원리 | 장점 |
|------|------|------|
| Isolation Forest | 랜덤 분할로 이상치 격리 | 고차원, 대용량에 강함 |
| HDBSCAN | 계층적 밀도 기반 클러스터링 | eps 불필요, 가변 밀도 적응 |
| LOF | 지역 밀도 비교 | 국소적 이상치에 강함 |

최종: 3개 모델 앙상블 (2개 이상 동의 시 이상 판정)  
해석: SHAP를 활용하여 Isolation Forest의 피처별 기여도를 분석한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import sys
sys.path.append('..')

from src.models import (
    detect_isolation_forest, detect_lof, detect_hdbscan,
    ensemble_anomaly, prepare_features, FEATURE_COLS
)
from src.explainer import compute_shap_values, get_top_anomaly_explanations

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

In [2]:
df = pd.read_parquet('../data/ais_featured.parquet')
print(f'{len(df):,} rows, {df["MMSI"].nunique()} vessels')

# 결측치 제거 (피처 컬럼 기준)
df_model = df.dropna(subset=FEATURE_COLS).copy()
print(f'모델 입력: {len(df_model):,} rows (결측 제거 후)')

630,026 rows, 500 vessels
모델 입력: 629,526 rows (결측 제거 후)


## 4.1 Isolation Forest

In [ ]:
df_model, if_model = detect_isolation_forest(df_model, contamination=0.05)

n_anomaly = df_model['anomaly_if'].sum()
print(f'Isolation Forest 이상 탐지: {n_anomaly:,} ({n_anomaly/len(df_model)*100:.2f}%)')

## 4.2 Local Outlier Factor

In [ ]:
df_model, lof_model = detect_lof(df_model, contamination=0.05, n_neighbors=30)

n_anomaly = df_model['anomaly_lof'].sum()
print(f'LOF 이상 탐지: {n_anomaly:,} ({n_anomaly/len(df_model)*100:.2f}%)')

## 4.3 HDBSCAN

DBSCAN과 달리 eps 파라미터가 불필요하며, 가변 밀도 클러스터에 적응적으로 동작한다.  
샘플링 없이 전체 데이터에 대해 직접 클러스터링하여, noise label(-1)을 이상으로 판정한다.

In [ ]:
df_model, hdb_model = detect_hdbscan(df_model, min_cluster_size=50, min_samples=10)

n_anomaly = df_model['anomaly_hdbscan'].sum()
print(f'HDBSCAN 이상 탐지: {n_anomaly:,} ({n_anomaly/len(df_model)*100:.2f}%)')

## 4.4 모델 간 비교

In [ ]:
anomaly_cols = ['anomaly_if', 'anomaly_lof', 'anomaly_hdbscan']
model_names = ['Isolation Forest', 'LOF', 'HDBSCAN']

# 모델별 이상 탐지 수
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = [df_model[c].sum() for c in anomaly_cols]
axes[0].bar(model_names, counts, color=['steelblue', 'coral', 'green'], alpha=0.8)
axes[0].set_ylabel('Anomaly Count')
axes[0].set_title('Anomalies Detected by Each Model')
for i, v in enumerate(counts):
    axes[0].text(i, v + 100, f'{v:,}', ha='center')

# 모델 간 합의
agreement = df_model[anomaly_cols].sum(axis=1).value_counts().sort_index()
axes[1].bar(agreement.index, agreement.values, color='purple', alpha=0.7)
axes[1].set_xlabel('Number of Models Agreeing (Anomaly)')
axes[1].set_ylabel('Record Count')
axes[1].set_title('Model Agreement Distribution')
axes[1].set_xticks([0, 1, 2, 3])

plt.tight_layout()
plt.savefig('../results/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.5 앙상블 (최종 판정)

In [7]:
df_model = ensemble_anomaly(df_model, threshold=2)

n_final = df_model['anomaly_final'].sum()
n_vessels = df_model[df_model['anomaly_final'] == 1]['MMSI'].nunique()

print(f'=== 최종 앙상블 결과 ===')
print(f'이상 레코드: {n_final:,} / {len(df_model):,} ({n_final/len(df_model)*100:.2f}%)')
print(f'이상 선박: {n_vessels} / {df_model["MMSI"].nunique()}')

=== 최종 앙상블 결과 ===
이상 레코드: 4,907 / 629,526 (0.78%)
이상 선박: 479 / 500


## 4.6 이상 탐지 결과 분석

In [ ]:
# 정상 vs 이상 피처 비교
normal = df_model[df_model['anomaly_final'] == 0]
anomaly = df_model[df_model['anomaly_final'] == 1]

print('=== 정상 vs 이상 피처 평균 비교 ===')
comparison = pd.DataFrame({
    'Normal': normal[FEATURE_COLS].mean(),
    'Anomaly': anomaly[FEATURE_COLS].mean(),
    'Ratio': anomaly[FEATURE_COLS].mean() / normal[FEATURE_COLS].mean().replace(0, 1)
})
print(comparison.round(2))

In [ ]:
# 피처별 정상 vs 이상 분포 비교
plot_features = ['SOG', 'speed_deviation', 'acceleration', 'course_change',
                 'heading_cog_diff', 'signal_gap_sec', 'stop_duration_min']
n_cols = 3
n_rows = (len(plot_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for idx, col in enumerate(plot_features):
    ax = axes[idx]
    clip_val = normal[col].quantile(0.99)
    normal[col].clip(upper=clip_val).hist(bins=40, ax=ax, alpha=0.5, color='blue', label='Normal', density=True)
    anomaly[col].clip(upper=clip_val).hist(bins=40, ax=ax, alpha=0.5, color='red', label='Anomaly', density=True)
    ax.set_title(col)
    ax.legend()

# 빈 subplot 제거
for idx in range(len(plot_features), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig('../results/figures/normal_vs_anomaly.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.6.1 SHAP 기반 피처 기여도 분석

Permutation Importance 대신 SHAP(SHapley Additive exPlanations)을 활용하여
각 피처가 이상 판정에 미치는 기여도를 정밀하게 분석한다.

- **Summary Plot**: 전체 피처의 SHAP 값 분포 → 어떤 피처가 이상 판정에 가장 중요한지
- **Bar Plot**: 평균 |SHAP| 기반 피처 중요도 순위
- **Waterfall Plot**: 개별 이상 레코드가 왜 이상으로 판정됐는지 분해

In [ ]:
# SHAP 값 계산 (5000 샘플)
shap_values, X_sample = compute_shap_values(if_model, df_model, max_samples=5000)

# Summary Plot
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_values, feature_names=FEATURE_COLS, max_display=10, show=False)
plt.tight_layout()
plt.savefig('../results/figures/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bar Plot - 피처 중요도 순위
fig, ax = plt.subplots(figsize=(10, 6))
shap.plots.bar(shap_values, max_display=10, show=False)
plt.tight_layout()
plt.savefig('../results/figures/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Waterfall Plot - 가장 이상한 레코드의 판정 근거
abs_shap = np.abs(shap_values.values).sum(axis=1)
top_idx = np.argmax(abs_shap)

print(f'가장 높은 SHAP 총합 레코드 (index={top_idx}):')
fig, ax = plt.subplots(figsize=(10, 5))
shap.plots.waterfall(shap_values[top_idx], show=False)
plt.tight_layout()
plt.savefig('../results/figures/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 10 이상 레코드의 주요 원인 피처
top_explanations = get_top_anomaly_explanations(shap_values, df_model, top_n=10)
print('=== SHAP 기반 이상 레코드 Top 10 ===')
print(top_explanations.to_string(index=False))

## 4.6.2 contamination 민감도 분석

contamination(오염 비율)을 바꾸면 결과가 얼마나 달라지는지 확인한다.

In [ ]:
# contamination 변화에 따른 앙상블 이상 탐지 수 변화
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

X, _ = prepare_features(df_model)

contamination_values = [0.01, 0.03, 0.05, 0.07, 0.10]
results = []

# HDBSCAN은 contamination 파라미터가 없으므로 고정 결과 사용
hdbscan_pred = df_model['anomaly_hdbscan'].values

for cont in contamination_values:
    if_model_s = IsolationForest(contamination=cont, random_state=42, n_jobs=-1)
    if_pred = (if_model_s.fit_predict(X) == -1).astype(int)

    lof_model_s = LocalOutlierFactor(contamination=cont, n_neighbors=30, n_jobs=-1)
    lof_pred = (lof_model_s.fit_predict(X) == -1).astype(int)

    ensemble = (if_pred + lof_pred + hdbscan_pred) >= 2
    results.append({
        'contamination': cont,
        'IF': if_pred.sum(),
        'LOF': lof_pred.sum(),
        'HDBSCAN': hdbscan_pred.sum(),
        'Ensemble (2/3)': ensemble.sum(),
        'Ensemble %': f'{ensemble.sum() / len(X) * 100:.2f}%'
    })

sens_df = pd.DataFrame(results)
print(sens_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot([r['contamination'] for r in results],
        [r['IF'] for r in results], 'o-', label='Isolation Forest', color='steelblue')
ax.plot([r['contamination'] for r in results],
        [r['LOF'] for r in results], 's-', label='LOF', color='coral')
ax.axhline(y=hdbscan_pred.sum(), color='green', linestyle='--', alpha=0.7, label=f'HDBSCAN (fixed: {hdbscan_pred.sum():,})')
ax.plot([r['contamination'] for r in results],
        [r['Ensemble (2/3)'] for r in results], 'D-', label='Ensemble (2/3)', color='purple', linewidth=2)

ax.set_xlabel('Contamination Rate')
ax.set_ylabel('Anomaly Count')
ax.set_title('Sensitivity Analysis: contamination vs Anomaly Count')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/sensitivity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.7 결과 저장

In [13]:
df_model.to_parquet('../data/ais_results.parquet', index=False)

import os
fsize = os.path.getsize('../data/ais_results.parquet') / 1e6
print(f'저장 완료: data/ais_results.parquet ({fsize:.1f} MB)')

저장 완료: data/ais_results.parquet (10.7 MB)
